# Global Fishing Watch raw data download

This script will download raw data from the global fishing watch dataset in BigQuery.

It allows to choose date, region and type of data.

We will use it to download the fishing efforts through 2020 - 2024 on Drafting longline vessels, to overlay shark species of interest distribution.

This project is made alongside SharkProject International.

In [ ]:
from google.colab import auth
from google.cloud import bigquery
from google.colab import files
import pandas as pd

print("Authenticating user...")
auth.authenticate_user()

# Initialize BigQuery client
# Replace 'YOUR_PROJECT_ID' with your active Google Cloud Project ID
project_id = 'YOUR_PROJECT_ID'
client = bigquery.Client(project=project_id)

# Execute query on GFW v3 database using the exact schema
query = """
SELECT
  cell_ll_lat AS lat_grid,
  cell_ll_lon AS lon_grid,
  SUM(fishing_hours) AS total_fishing_hours
FROM
  `global-fishing-watch.fishing_effort_v3.fleet_monthly_10_v3`
WHERE
  -- Temporal filter (Using the dedicated 'year' column)
  year BETWEEN 2020 AND 2024

  -- Gear type filter (Pelagic longlines)
  AND geartype = 'drifting_longlines'

  -- Spatial filter (NE Atlantic & Mediterranean)
  AND cell_ll_lat BETWEEN 25.0 AND 60.0
  AND cell_ll_lon BETWEEN -35.0 AND 30.0

  -- Exclude zero effort
  AND fishing_hours > 0
GROUP BY
  lat_grid,
  lon_grid
ORDER BY
  total_fishing_hours DESC
"""

print("Executing BigQuery...")

# Run the query and convert to DataFrame
df = client.query(query).to_dataframe()
print(f"Query successful. Extracted {len(df)} grid cells.")

# Export and download
output_filename = 'GFW_Longlines_Europe_2020_2024_Final.csv'
df.to_csv(output_filename, index=False)
print(f"Saved to '{output_filename}'. Initiating download...")

# Download to your local machine
files.download(output_filename)